# Emulator Inversion: Recovering Parameters from Observations

Given observed distances or power spectra, can we recover the cosmological
parameters that produced them? This notebook demonstrates the emulator
inversion interface — finding inputs from target outputs.

### Steps
1. Train a GP emulator on `comoving_angular_distance`
2. Generate synthetic "observations" from known Planck parameters
3. Recover a single parameter (Omega_c) with the others fixed
4. Recover two parameters simultaneously (Omega_c, h)
5. Compare GP (uncertainty-weighted) vs generic (scipy) inversion
6. Demonstrate PyTorch gradient-based inversion

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tissage_cosmique.computations.distances import comoving_angular_distance
from tissage_cosmique.emulators import (
    GPEmulator,
    build_training_data,
    params_to_feature_matrix,
    invert_minimize,
)

## 1. Train a GP emulator

In [ ]:
PARAM_NAMES = ["Omega_c", "h", "sigma8"]
FIXED = dict(Omega_b=0.0486, n_s=0.9667, Omega_k=0.0, w0=-1.0, wa=0.0)
a_grid = np.linspace(0.2, 0.8, 30)

BOUNDS = {
    "Omega_c": (0.20, 0.35),
    "h": (0.60, 0.80),
    "sigma8": (0.70, 0.90),
}

rng = np.random.default_rng(42)
train_samples = [
    {"Omega_c": rng.uniform(*BOUNDS["Omega_c"]),
     "h": rng.uniform(*BOUNDS["h"]),
     "sigma8": rng.uniform(*BOUNDS["sigma8"]), **FIXED}
    for _ in range(50)
]

X, y = build_training_data(comoving_angular_distance, train_samples, a_grid, param_names=PARAM_NAMES)
emu = GPEmulator(feature_names=PARAM_NAMES + ["a"])
emu.fit(X, y)
print(f"Emulator trained: R2={emu.metadata['training_score']:.6f}")

## 2. Generate synthetic observations

We use known Planck 2018 parameters and compute the "true" distances.
The inversion should recover these parameters.

In [ ]:
PLANCK = dict(Omega_c=0.2589, h=0.6774, sigma8=0.8159, **FIXED)

y_observed = comoving_angular_distance(PLANCK, a_grid)
print(f"Observed distances at {len(a_grid)} scale factors")
print(f"Range: [{y_observed.min():.0f}, {y_observed.max():.0f}] Mpc")

## 3. Recover a single parameter: Omega_c

Fix h and sigma8 at their true values, solve for Omega_c.

In [ ]:
result = emu.invert(
    y_target=y_observed,
    free_params=["Omega_c"],
    fixed_params={"h": 0.6774, "sigma8": 0.8159, "a": a_grid},
    x0={"Omega_c": 0.27},
    bounds={"Omega_c": BOUNDS["Omega_c"]},
)

print(f"True Omega_c:      {PLANCK['Omega_c']:.4f}")
print(f"Recovered Omega_c: {result.x_solution['Omega_c']:.4f}")
print(f"Error:             {abs(result.x_solution['Omega_c'] - PLANCK['Omega_c']):.5f}")
print(f"Residual:          {result.residual:.6f}")
print(f"Success:           {result.success}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(a_grid, y_observed, "k-", lw=2, label="Observed (Planck)")
ax.plot(a_grid, result.y_predicted, "r--", lw=1.5, label="Inverted prediction")
ax.set_xlabel("Scale factor a")
ax.set_ylabel("Distance [Mpc]")
ax.set_title(f"Single-param inversion: $\\Omega_c$ = {result.x_solution['Omega_c']:.4f}")
ax.legend()

ax = axes[1]
residual = result.y_predicted - y_observed
ax.plot(a_grid, residual, "b-", lw=1.5)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Scale factor a")
ax.set_ylabel("Residual [Mpc]")
ax.set_title(f"Residual (RMSE = {np.sqrt(np.mean(residual**2)):.2f} Mpc)")

plt.tight_layout()
plt.show()

## 4. Recover two parameters: Omega_c and h

Fix only sigma8, solve for both Omega_c and h simultaneously.

In [ ]:
result_2p = emu.invert(
    y_target=y_observed,
    free_params=["Omega_c", "h"],
    fixed_params={"sigma8": 0.8159, "a": a_grid},
    x0={"Omega_c": 0.27, "h": 0.68},
    bounds={"Omega_c": BOUNDS["Omega_c"], "h": BOUNDS["h"]},
)

print(f"{'Parameter':<12} {'True':>8} {'Recovered':>10} {'Error':>10}")
print("-" * 42)
for p in ["Omega_c", "h"]:
    true_val = PLANCK[p]
    rec_val = result_2p.x_solution[p]
    print(f"{p:<12} {true_val:>8.4f} {rec_val:>10.4f} {abs(rec_val - true_val):>10.5f}")
print(f"\nResidual: {result_2p.residual:.6f}")
print(f"Success:  {result_2p.success}")

## 5. Scan the objective landscape

Visualize the loss surface to see why the optimizer converges.

In [ ]:
omega_c_range = np.linspace(*BOUNDS["Omega_c"], 40)
h_range = np.linspace(*BOUNDS["h"], 40)
loss_grid = np.zeros((len(h_range), len(omega_c_range)))

for i, h_val in enumerate(h_range):
    for j, oc_val in enumerate(omega_c_range):
        test_params = {"Omega_c": oc_val, "h": h_val, "sigma8": 0.8159}
        X_pred = params_to_feature_matrix(test_params, a_grid, param_names=PARAM_NAMES)
        y_pred = emu.predict(X_pred)
        loss_grid[i, j] = np.mean(((y_pred - y_observed) / y_observed.mean()) ** 2)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.contourf(omega_c_range, h_range, np.log10(loss_grid + 1e-12), levels=30, cmap="viridis_r")
cb = plt.colorbar(im, ax=ax)
cb.set_label("log10(loss)")

ax.plot(PLANCK["Omega_c"], PLANCK["h"], "w*", markersize=15, label="True (Planck)")
ax.plot(result_2p.x_solution["Omega_c"], result_2p.x_solution["h"],
        "rx", markersize=12, mew=3, label="Recovered")
ax.plot(0.27, 0.68, "g+", markersize=12, mew=2, label="Initial guess")

ax.set_xlabel("$\\Omega_c$")
ax.set_ylabel("h")
ax.set_title("Inversion loss landscape")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Compare GP vs generic inversion

The GP backend uses uncertainty-weighted minimization while the generic
``invert_minimize`` uses unweighted least squares.

In [ ]:
# GP inversion (uncertainty-weighted)
gp_result = emu.invert(
    y_target=y_observed,
    free_params=["Omega_c", "h"],
    fixed_params={"sigma8": 0.8159, "a": a_grid},
    x0={"Omega_c": 0.27, "h": 0.68},
    bounds={"Omega_c": BOUNDS["Omega_c"], "h": BOUNDS["h"]},
)

# Generic inversion (unweighted scipy.optimize)
generic_result = invert_minimize(
    emu, y_observed,
    free_params=["Omega_c", "h"],
    fixed_params={"sigma8": 0.8159, "a": a_grid},
    x0={"Omega_c": 0.27, "h": 0.68},
    bounds={"Omega_c": BOUNDS["Omega_c"], "h": BOUNDS["h"]},
)

print(f"{'Method':<25} {'Omega_c':>10} {'h':>10} {'Residual':>10}")
print("-" * 57)
print(f"{'True (Planck)':<25} {PLANCK['Omega_c']:>10.4f} {PLANCK['h']:>10.4f}")
print(f"{'GP (uncertainty-weighted)':<25} {gp_result.x_solution['Omega_c']:>10.4f} "
      f"{gp_result.x_solution['h']:>10.4f} {gp_result.residual:>10.6f}")
print(f"{'Generic (scipy)':<25} {generic_result.x_solution['Omega_c']:>10.4f} "
      f"{generic_result.x_solution['h']:>10.4f} {generic_result.residual:>10.6f}")

## 7. PyTorch gradient-based inversion

Neural network emulators can invert via backpropagation — optimizing
the input parameters directly through the network.

In [ ]:
from tissage_cosmique.emulators import PyTorchEmulator

pt_emu = PyTorchEmulator(
    feature_names=PARAM_NAMES + ["a"], hidden_layers=[64, 64], n_epochs=500, seed=42,
)
pt_emu.fit(X, y)
print(f"PyTorch emulator trained: loss={pt_emu.metadata['training_loss']:.4f}")

pt_result = pt_emu.invert(
    y_target=y_observed,
    free_params=["Omega_c", "h"],
    fixed_params={"sigma8": 0.8159, "a": a_grid},
    x0={"Omega_c": 0.27, "h": 0.68},
    bounds={"Omega_c": BOUNDS["Omega_c"], "h": BOUNDS["h"]},
    n_steps=1000,
)

print(f"\nPyTorch inversion:")
print(f"  Omega_c: {pt_result.x_solution['Omega_c']:.4f} (true: {PLANCK['Omega_c']:.4f})")
print(f"  h:       {pt_result.x_solution['h']:.4f} (true: {PLANCK['h']:.4f})")
print(f"  Residual: {pt_result.residual:.6f}")

## 8. Sensitivity: how do recovered parameters change with initial guess?

Run the inversion from different starting points to check robustness.

In [ ]:
starts = [
    {"Omega_c": 0.22, "h": 0.62},
    {"Omega_c": 0.27, "h": 0.68},
    {"Omega_c": 0.30, "h": 0.75},
    {"Omega_c": 0.34, "h": 0.62},
    {"Omega_c": 0.22, "h": 0.78},
]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.contourf(omega_c_range, h_range, np.log10(loss_grid + 1e-12), levels=30, cmap="viridis_r")
plt.colorbar(im, ax=ax, label="log10(loss)")

for i, x0 in enumerate(starts):
    r = invert_minimize(
        emu, y_observed,
        free_params=["Omega_c", "h"],
        fixed_params={"sigma8": 0.8159, "a": a_grid},
        x0=x0,
        bounds={"Omega_c": BOUNDS["Omega_c"], "h": BOUNDS["h"]},
    )
    ax.annotate(
        "", xy=(r.x_solution["Omega_c"], r.x_solution["h"]),
        xytext=(x0["Omega_c"], x0["h"]),
        arrowprops=dict(arrowstyle="->", color=f"C{i}", lw=1.5),
    )
    ax.plot(x0["Omega_c"], x0["h"], "o", color=f"C{i}", markersize=6)

ax.plot(PLANCK["Omega_c"], PLANCK["h"], "w*", markersize=15, label="True")
ax.set_xlabel("$\\Omega_c$")
ax.set_ylabel("h")
ax.set_title("Inversion from different starting points")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

The inversion interface recovers cosmological parameters from target outputs:

- **Single parameter**: exact recovery when the others are fixed
- **Multiple parameters**: joint recovery with good accuracy when the loss landscape is well-behaved
- **Backend-specific methods**: GP uses uncertainty weighting, PyTorch uses gradient descent
- **Robust to initial guess**: the optimizer converges from different starting points

The `invert()` method works identically on all emulator backends — the generic
scipy minimizer handles any backend, while GP/PyTorch/TF/PySR provide
specialized implementations when available.